In [1]:
# Cell 1 — Setup
import sys, os
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display, HTML

PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from data_loader import (
    EMOTION_LABELS, STRESS_MAP, EMOTION_VALENCE,
    DATA_PROC, CNN_WEIGHT, PHYSIO_WEIGHT,
    compute_emotion_stress_score, fused_score_to_label,
)
from model        import load_model, MODEL_PATH
from realtime     import preprocess_face, JITAI_SUGGESTIONS, fuse_scores
from physio_branch import PhysioBranch

print(f'TF: {tf.__version__}')
print(f'Fusion: CNN {CNN_WEIGHT*100:.0f}% + Physio {PHYSIO_WEIGHT*100:.0f}%')

2026-04-11 19:20:35.436174: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-11 19:20:35.980543: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-11 19:20:35.980716: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-11 19:20:36.060176: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-11 19:20:36.244546: I tensorflow/core/platform/cpu_feature_guar

mediapipe not installed. Run: pip install mediapipe
   Physio branch will return zero scores (CNN-only mode).
TF: 2.15.0
Fusion: CNN 20% + Physio 80%


In [2]:
# Cell 2 — Load CNN model
model = load_model(MODEL_PATH)
print(f'CNN model loaded: {MODEL_PATH.name}')
print(f'Output: {model.output_shape}  (7-class softmax)')

Loading CNN model from /home/ariba/Projects/stress_detection_project - Multi Factor/models/stress_emotion_model.keras


2026-04-11 19:20:45.710970: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-11 19:20:46.490456: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-11 19:20:46.490526: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-11 19:20:46.495896: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-11 19:20:46.495962: I external/local_xla/xla/stream_executor

CNN model loaded: stress_emotion_model.keras
Output: (None, 7)  (7-class softmax)


In [3]:
# Cell 3 — Demo: CNN Branch on test image
test_images = np.load(DATA_PROC / 'test_images_proc.npy')
test_labels = np.load(DATA_PROC / 'test_labels_ohe.npy')

# Pick 5 samples
sample_idxs = np.random.choice(len(test_images), 5, replace=False)
print('CNN Branch predictions on test samples:')
print(f'{"True":<12} {"Predicted":<12} {"Emotion Score":>14} {"Conf":>6}')
print('-' * 50)
for idx in sample_idxs:
    img     = test_images[idx:idx+1]
    true_cls= int(test_labels[idx].argmax())
    preds   = model.predict(img, verbose=0)[0]
    pred_cls= int(preds.argmax())
    conf    = float(preds.max())
    em_score= compute_emotion_stress_score(pred_cls, conf)
    true_lbl= EMOTION_LABELS[true_cls]
    pred_lbl= EMOTION_LABELS[pred_cls]
    ok = '✅' if true_cls == pred_cls else '❌'
    print(f'{ok} {true_lbl:<12} {pred_lbl:<12} {em_score:>14.3f} {conf*100:>5.1f}%')

CNN Branch predictions on test samples:
True         Predicted     Emotion Score   Conf
--------------------------------------------------


2026-04-11 19:20:50.055892: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


✅ Fear         Fear                  0.390  39.0%
❌ Fear         Neutral               0.000  76.9%
✅ Happy        Happy                 0.000  96.3%
✅ Sad          Sad                   0.472  94.5%
❌ Neutral      Happy                 0.000  58.1%


In [4]:
# Cell 4 — Physio Branch demo (simulated signals)
print('Physio Branch — Signal Simulation')
print('=' * 55)
print('(In real-time mode, signals come from MediaPipe 478 landmarks)')
print()

simulated_scenarios = [
    {'name': 'Relaxed (Low Stress)',   'blink_rate': 0.1, 'eye_openness': 0.1,
     'gaze_jitter': 0.05, 'head_motion': 0.05, 'brow_tension': 0.05, 'jaw_tension': 0.05},
    {'name': 'Moderate (Med Stress)',  'blink_rate': 0.4, 'eye_openness': 0.4,
     'gaze_jitter': 0.3,  'head_motion': 0.3,  'brow_tension': 0.3,  'jaw_tension': 0.3},
    {'name': 'Stressed (High Stress)', 'blink_rate': 0.9, 'eye_openness': 0.8,
     'gaze_jitter': 0.7,  'head_motion': 0.7,  'brow_tension': 0.7,  'jaw_tension': 0.8},
]
weights = {'blink_rate': 0.25, 'eye_openness': 0.16, 'gaze_jitter': 0.16,
           'head_motion': 0.12, 'brow_tension': 0.08, 'jaw_tension': 0.08}
total_w = sum(weights.values())

for scenario in simulated_scenarios:
    raw = sum(weights[k] * scenario[k] for k in weights)
    physio_score = raw / total_w
    print(f'Scenario: {scenario["name"]}')
    print(f'  Physio score: {physio_score:.3f}')
    print(f'  Signals: blink={scenario["blink_rate"]:.2f} eye={scenario["eye_openness"]:.2f} '
          f'gaze={scenario["gaze_jitter"]:.2f}')
    print()

Physio Branch — Signal Simulation
(In real-time mode, signals come from MediaPipe 478 landmarks)

Scenario: Relaxed (Low Stress)
  Physio score: 0.074
  Signals: blink=0.10 eye=0.10 gaze=0.05

Scenario: Moderate (Med Stress)
  Physio score: 0.348
  Signals: blink=0.40 eye=0.40 gaze=0.30

Scenario: Stressed (High Stress)
  Physio score: 0.787
  Signals: blink=0.90 eye=0.80 gaze=0.70



In [5]:
# Cell 5 — Full dual-branch fusion demo
print('Full Dual-Branch Fusion Demo')
print('Formula: 0.20 × emotion_score + 0.80 × physio_score')
print('=' * 65)

# Sample 8 test images
idxs = np.random.choice(len(test_images), 8, replace=False)
physio_sims = np.random.uniform(0.1, 0.9, 8)  # simulated physio scores

print(f'{"Emotion":<12} {"E.Score":>7} {"Physio":>7} {"Fused":>7} {"Stress":<10}')
print('-' * 55)
for i, (idx, pscore) in enumerate(zip(idxs, physio_sims)):
    img      = test_images[idx:idx+1]
    preds    = model.predict(img, verbose=0)[0]
    cls      = int(preds.argmax())
    conf     = float(preds.max())
    emotion  = EMOTION_LABELS[cls]
    e_score  = compute_emotion_stress_score(cls, conf)
    fused, stress_label, _ = fuse_scores(e_score, pscore)
    icons = {'High': '🔴', 'Medium': '🟠', 'Low': '🟢'}
    print(f'{emotion:<12} {e_score:>7.3f} {pscore:>7.3f} {fused:>7.3f} '
          f'{icons[stress_label]} {stress_label}')

Full Dual-Branch Fusion Demo
Formula: 0.20 × emotion_score + 0.80 × physio_score
Emotion      E.Score  Physio   Fused Stress    
-------------------------------------------------------
Sad            0.447   0.141   0.202 🟢 Low
Fear           0.979   0.775   0.816 🔴 High
Neutral        0.000   0.482   0.386 🟠 Medium
Happy          0.000   0.372   0.297 🟢 Low
Fear           0.532   0.820   0.763 🔴 High
Surprise       0.424   0.434   0.432 🟠 Medium
Happy          0.000   0.173   0.138 🟢 Low
Angry          0.522   0.797   0.742 🔴 High


In [6]:
# Cell 6 — JITAI Engine demo
from realtime import JITAIEngine
print('JITAI Engine Demo (Just-In-Time Adaptive Interventions)')
print('=' * 55)

engine = JITAIEngine(window=10, threshold=7)

for level in ['High']*8 + ['Low']*3:
    result = engine.update(level)
    if result:
        print(f'\nJITAI Triggered (dominant: High):')
        print(f'   {result}')

print('\nSuggestions by stress level:')
for level in ['High', 'Medium', 'Low']:
    print(f'\n  {level}:')
    for s in JITAI_SUGGESTIONS[level]:
        print(f'    • {s}')

JITAI Engine Demo (Just-In-Time Adaptive Interventions)

JITAI Triggered (dominant: High):
   🧘 Try 4-7-8 breathing: inhale 4s, hold 7s, exhale 8s

Suggestions by stress level:

  High:
    • 🧘 Try 4-7-8 breathing: inhale 4s, hold 7s, exhale 8s
    • 💧 Drink a glass of water and step away from screen
    • 🚶 Take a 5-minute walk to reset your mind
    • 🎵 Listen to calming music for 3 minutes
    • ✍️  Write down what's stressing you right now

  Medium:
    • ☕ Take a short break (5-10 minutes)
    • 🌿 Look at something green for 20 seconds (20-20-20 rule)
    • 🤸 Do 5 neck rolls and shoulder shrugs
    • 📵 Silence notifications for the next 30 minutes

  Low:
    • ✅ Great focus! Keep going!
    • 🌟 You're in a calm state — ideal for creative work
    • 🎯 Perfect time to tackle challenging tasks


In [7]:
# Cell 7 — Launch instructions
print('🚀 How to run the full system:')
print()
print('Option 1: Streamlit Web App')
print('  streamlit run app/streamlit_app.py')
print('  → Open http://localhost:8501')
print()
print('Option 2: OpenCV Real-Time Window')
print('  python src/realtime.py')
print('  → Webcam window opens')
print('  → Keys: Q=Quit | S=Suggestion | D=Debug panel')
print()
print('Debug panel (press D):')
print('  Shows live MediaPipe physio signal bars:')
print('  Blink Rate ×0.25, Eye Openness ×0.16, Gaze Jitter ×0.16')
print('  Head Motion ×0.12, Brow Tension ×0.08, Jaw Tension ×0.08')
print()


🚀 How to run the full system:

Option 1: Streamlit Web App
  streamlit run app/streamlit_app.py
  → Open http://localhost:8501

Option 2: OpenCV Real-Time Window
  python src/realtime.py
  → Webcam window opens
  → Keys: Q=Quit | S=Suggestion | D=Debug panel

Debug panel (press D):
  Shows live MediaPipe physio signal bars:
  Blink Rate ×0.25, Eye Openness ×0.16, Gaze Jitter ×0.16
  Head Motion ×0.12, Brow Tension ×0.08, Jaw Tension ×0.08

